<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 警告：此文件不属于训练营的教学步骤。
# 这是一个中高级别的示例，展示了您将学习到的内容。
# 如果您正在参加训练营以学习 CHISEL，请不要从这里开始。
# 请从 [Scala 简介](1_intro_to_scala.ipynb) 开始。

# Chisel 演示
**下一步：[Scala 简介](1_intro_to_scala.ipynb)**

欢迎！也许您是一位感兴趣的学生，听人说起过“Chisel”这个名字；或者您是一位经验丰富的硬件设计老手，经理要求您探索 Chisel 作为一种新的 HDL 替代方案。无论哪种方式，如果您是 Chisel 的新手，您都希望尽快了解它到底有什么了不起。别再犹豫了——让我们看看 Chisel 能提供什么！

## 设置
在开始之前，我们需要下载并导入演示所需的依赖项。

**请通过按键盘上的 SHIFT+ENTER 或菜单中的“运行”按钮来运行以下两个单元格块。**

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.iotesters.{ChiselFlatSpec, Driver, PeekPokeTester}

## 硬件生成器：RTL 的类型安全元编程

所有硬件描述语言都支持编写 RTL 设计的单个实例——Chisel 也不例外。
事实上，大多数 Verilog/VHDL 数字逻辑设计都可以直接转录到 Chisel 中！
虽然 Chisel 提供了其他很棒的功能（我们稍后会介绍），但我们想强调的是，切换到 Chisel 的用户将保留与任何其他硬件语言完全相同的设计控制程度。

以下是一个以 FIR 滤波器风格实现的 3 点移动平均的示例。

<img src="images/demo_fir_filter.svg" width="512" />

Chisel 提供了与可综合 Verilog 类似的基本原语，并且*可以*这样使用！运行下一个单元格来声明我们的 Chisel 模块。

In [ ]:
// 以 FIR 滤波器风格实现的 3 点移动平均
class MovingAverage3(bitWidth: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt(bitWidth.W))
  })

  val z1 = RegNext(io.in) // 创建一个寄存器，其输入连接到参数 io.in
  val z2 = RegNext(z1)    // 创建一个寄存器，其输入连接到参数 z1

  io.out := (io.in * 1.U) + (z1 * 1.U) + (z2 * 1.U) // `1.U` 是值为 1 的无符号文字
}

定义 `class MovingAverage3` 后，让我们实例化它并查看其结构：

In [ ]:
// 与之前相同的 3 点移动平均滤波器
visualize(() => new MovingAverage3(8))

在这个 Chisel 实例的可视化中，左侧是输入，金色的是 z1 和 z2 寄存器。寄存器和 io_in 都乘以它们的系数，然后依次相加。`tail` 和 `bits` 元素用于防止加法结果增长过大。

您现在可能会问：“哦，这很好——您可以在 Chisel 中完成 Verilog 中的工作，但那我为什么要使用 Chisel 呢？”

我们很高兴您这么问！Chisel 的真正威力来自于创建**生成器，而不是实例**的能力。假设我们不仅仅想要一个 `MovingAverage3` 模块，而是想要创建一个通用的 `FIRFilter` 模块，该模块由一系列系数参数化。

下面我们重写了 `MovingAverage3` 以接受一系列系数。系数的数量将决定滤波器的大小。

In [ ]:
// 由卷积系数参数化的广义 FIR 滤波器
class FirFilter(bitWidth: Int, coeffs: Seq[UInt]) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt())
  })
  // 创建串行输入、并行输出移位寄存器
  val zs = Reg(Vec(coeffs.length, UInt(bitWidth.W)))
  zs(0) := io.in
  for (i <- 1 until coeffs.length) {
    zs(i) := zs(i-1)
  }

  // 进行乘法运算
  val products = VecInit.tabulate(coeffs.length)(i => zs(i) * coeffs(i))

  // 将乘积相加
  io.out := products.reduce(_ +& _)
}

现在，通过在实例化期间更改我们的 `coeffs` 参数，我们的 `FIRFilter` 模块可以用于实例化无数个不同的硬件模块！下面我们创建三个不同的 `FIRFiler` 实例

In [ ]:
// 与之前相同的 3 点移动平均滤波器
visualize(() => new FirFilter(8, Seq(1.U, 1.U, 1.U)))

In [ ]:
// 作为 FIR 滤波器的 1 周期延迟
visualize(() => new FirFilter(8, Seq(0.U, 1.U)))

In [ ]:
// 具有三角形脉冲响应的 5 点 FIR 滤波器
visualize(() => new FirFilter(8, Seq(1.U, 2.U, 3.U, 2.U, 1.U)))

如果没有这种强大的参数化功能，我们将需要更多的模块定义，可能每个 FIR 滤波器都需要一个。理想情况下，我们希望我们的生成器是（1）可组合的，（2）强大的，并且（3）能够对生成的设计进行细粒度控制。

Chisel 的优势在于您如何使用它，而不在于语言本身。
如果您决定编写实例而不是生成器，那么与 Verilog 相比，您将看到 Chisel 的优势较少。
但是，如果您花时间学习如何编写生成器，那么 Chisel 的强大功能将变得显而易见，您会发现自己再也回不去编写 Verilog 了。
学习编写生成器是困难的，但我们希望本教程能为您铺平道路，使您成为一名更好的硬件设计师、程序员和思考者！

---
# 全部完成！

[返回顶部。](#top)